In [ ]:
import pandas as pd
import sys
import copy
import torch 
import torch.nn as nn
print(sys.executable)
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
print(sklearn.__version__)
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("../data/raw/alzdata.csv")

df.head()
df.shape
 
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")
    print(df[col].min(), df[col].max())
df.dtypes

df = df.drop(columns = ["PatientID", "DoctorInCharge"])
df.isnull().sum()
(df.isnull().sum()/len(df))*100
df.duplicated().sum()
df = df.drop_duplicates()
df.describe()

In [ ]:
ind = df.drop(columns=["Diagnosis"])
dep = df["Diagnosis"]

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    ind, dep, test_size=0.2, random_state=42, stratify=dep
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)

model = nn.Sequential(
    nn.Linear(X_train.shape[1], 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(X_test)).numpy().ravel()

predictions = (probabilities >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, probabilities))
print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)

torch.manual_seed(42)
np.random.seed(42)

X = ind.to_numpy(dtype=np.float32)
y = np.asarray(dep).astype(int)

assert set(np.unique(y)).issubset({0, 1}), "Target must be binary 0/1."

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50,
    random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val_np = scaler.transform(X_val_raw).astype(np.float32)
X_test_np = scaler.transform(X_test_raw).astype(np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def tensor_x(x):
    return torch.tensor(x, dtype=torch.float32, device=device)

def tensor_y(y):
    return torch.tensor(
        np.asarray(y), dtype=torch.float32, device=device
    ).reshape(-1, 1)

X_train_t = tensor_x(X_train_np)
X_val_t = tensor_x(X_val_np)
X_test_t = tensor_x(X_test_np)

y_train_t = tensor_y(y_train)
y_val_t = tensor_y(y_val)

# Account for class imbalance in the training loss
positive = max(int(y_train.sum()), 1)
negative = max(len(y_train) - int(y_train.sum()), 1)

pos_weight = torch.tensor(
    [negative / positive], dtype=torch.float32, device=device
)

In [ ]:
def train_model(model, learning_rate=0.001,
                weight_decay=0.001, epochs=200, patience=20):

    model = model.to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    wait = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(X_train_t)
        loss = loss_fn(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = loss_fn(val_logits, y_val_t).item()

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    model.load_state_dict(best_state)
    return model


def predict_prob(model, X_tensor):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X_tensor)).cpu().numpy().ravel()


def best_threshold(y_true, probabilities):
    # Choose threshold on validation data only
    thresholds = np.arange(0.10, 0.91, 0.05)
    scores = [
        f1_score(y_true, (probabilities >= t).astype(int),
                 zero_division=0)
        for t in thresholds
    ]
    return float(thresholds[int(np.argmax(scores))])


def evaluate_model(name, model, threshold=0.5):
    probabilities = predict_prob(model, X_test_t)
    predictions = (probabilities >= threshold).astype(int)

    results = {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "PR-AUC": average_precision_score(y_test, probabilities),
        "Accuracy": accuracy_score(y_test, predictions),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        )
    }

    print(f"\n{name}")
    print(classification_report(
        y_test, predictions, zero_division=0
    ))
    return results

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32),
        dropout=0.2):
        super().__init__()
        layers = []
        previous = input_dim
        for width in hidden_dims:
            layers.extend([
                nn.Linear(previous, width),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            previous = width
        layers.append(nn.Linear(previous, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


configs = [
    {"hidden_dims": (32,), "dropout": 0.1,
     "lr": 0.001, "wd": 0.001},
    {"hidden_dims": (64, 32), "dropout": 0.2,
     "lr": 0.001, "wd": 0.001},
    {"hidden_dims": (128, 64), "dropout": 0.3,
     "lr": 0.0005, "wd": 0.01},
]

tuning_results = []
best_mlp = None
best_auc = -1

for config in configs:
    torch.manual_seed(42)

    candidate = MLP(
        input_dim=X_train_t.shape[1],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"]
    )

    candidate = train_model(
        candidate,
        learning_rate=config["lr"],
        weight_decay=config["wd"]
    )

    val_probs = predict_prob(candidate, X_val_t)
    val_auc = roc_auc_score(y_val, val_probs)

    tuning_results.append({
        **config,
        "Validation ROC-AUC": val_auc
    })

    if val_auc > best_auc:
        best_auc = val_auc
        best_mlp = copy.deepcopy(candidate)

tuning_table = pd.DataFrame(tuning_results)
print(tuning_table.sort_values(
    "Validation ROC-AUC", ascending=False
))

mlp_threshold = best_threshold(
    y_val, predict_prob(best_mlp, X_val_t)
)

print("Selected validation ROC-AUC:", best_auc)

In [ ]:
clin_cols = [
    "BMI",
    "Smoking",
    "AlcoholConsumption",
    "PhysicalActivity",
    "DietQuality",
    "SleepQuality",
    "FunctionalAssessment",
    "MemoryComplaints",
    "BehavioralProblems",
    "ADL",
    "Confusion",
    "Disorientation",
    "PersonalityChanges",
    "DifficultyCompletingTasks",
    "Forgetfulness"
]

dem_cols = [
    "Age",
    "Gender",
    "Ethnicity",
    "EducationLevel"
]
all_cols = list(ind.columns)

assert set(clin_cols).issubset(all_cols), \
    "Update clinical_cols with actual column names."
assert set(dem_cols).issubset(all_cols), \
    "Update demographic_cols with actual column names."
assert set(clin_cols).isdisjoint(dem_cols), \
    "A feature should not belong to both groups."
clinical_idx = [all_cols.index(c) for c in clin_cols]
demo_idx = [all_cols.index(c) for c in dem_cols]


In [ ]:
class DualEncoderMLP(nn.Module):
    def __init__(self, clinical_idx, demo_idx, dropout=0.2):
        super().__init__()
        self.clinical_idx = clinical_idx
        self.demo_idx = demo_idx
        self.clinical_encoder = nn.Sequential(
            nn.Linear(len(clinical_idx), 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.ReLU()
        )
        self.demo_encoder = nn.Sequential(
            nn.Linear(len(demo_idx), 16),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.predictor = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        clinical = x[:, self.clinical_idx]
        demographic = x[:, self.demo_idx]
        clinical_repr = self.clinical_encoder(clinical)
        demo_repr = self.demo_encoder(demographic)
        combined = torch.cat( [clinical_repr, demo_repr], dim=1)
        return self.predictor(combined)
torch.manual_seed(42)
dual_model = DualEncoderMLP(clinical_idx, demo_idx)
dual_model = train_model(
    dual_model,
    learning_rate=0.001,
    weight_decay=0.001
)
dual_threshold = best_threshold(
    y_val, predict_prob(dual_model, X_val_t)
)

In [ ]:
class AttentionMLP(nn.Module):
    def __init__(self, clinical_idx, demo_idx,
         embedding_dim=32, dropout=0.2):
        super().__init__()
        self.clinical_idx = clinical_idx
        self.demo_idx = demo_idx
        self.clinical_encoder = nn.Sequential(
            nn.Linear(len(clinical_idx), embedding_dim),
            nn.ReLU()
        )
        self.demo_encoder = nn.Sequential(
            nn.Linear(len(demo_idx), embedding_dim),
            nn.ReLU()
        )
        self.attention_score = nn.Linear(embedding_dim, 1)
        self.predictor = nn.Sequential(
            nn.Linear(embedding_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        clinical = self.clinical_encoder(
            x[:, self.clinical_idx]
        )
        demographic = self.demo_encoder(
            x[:, self.demo_idx]
        )
        tokens = torch.stack(
            [clinical, demographic], dim=1
        )
        scores = self.attention_score(tokens).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        combined = (
            tokens * weights.unsqueeze(-1)
        ).sum(dim=1)
        return self.predictor(combined)


torch.manual_seed(42)
attention_model = AttentionMLP(
    clinical_idx, demo_idx
)
attention_model = train_model(
    attention_model,
    learning_rate=0.001,
    weight_decay=0.001
)
attention_threshold = best_threshold(
    y_val, predict_prob(attention_model, X_val_t)
)

In [ ]:
attention_model.eval()
with torch.no_grad():
    tokens = torch.stack([
        attention_model.clinical_encoder(
            X_test_t[:, clinical_idx]
        ),
        attention_model.demo_encoder(
            X_test_t[:, demo_idx]
        )
    ], dim=1)
    weights = torch.softmax(
        attention_model.attention_score(tokens).squeeze(-1),
        dim=1
    ).cpu().numpy()
print("Mean clinical attention:", weights[:, 0].mean())
print("Mean demographic attention:", weights[:, 1].mean())

In [ ]:
class ClinicalTransformer(nn.Module):
    def __init__(self, clinical_idx, demo_idx,
                 d_model=32, dropout=0.2):
        super().__init__()
        self.clinical_idx = clinical_idx
        self.demo_idx = demo_idx
        self.clinical_encoder = nn.Linear(
            len(clinical_idx), d_model
        )
        self.demo_encoder = nn.Linear(
            len(demo_idx), d_model
        )
        self.position = nn.Parameter(
            torch.zeros(1, 2, d_model)
        )
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=64,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers=2
        )
        self.predictor = nn.Sequential(
            nn.Linear(d_model * 2, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        clinical = self.clinical_encoder(
            x[:, self.clinical_idx]
        )
        demographic = self.demo_encoder(
            x[:, self.demo_idx]
        )
        tokens = torch.stack(
            [clinical, demographic], dim=1
        )
        tokens = tokens + self.position
        encoded = self.transformer(tokens)
        flattened = encoded.reshape(encoded.shape[0], -1)
        return self.predictor(flattened)
torch.manual_seed(42)
transformer_model = ClinicalTransformer(
    clinical_idx, demo_idx
)
transformer_model = train_model(
    transformer_model,
    learning_rate=0.0005,
    weight_decay=0.001
)
transformer_threshold = best_threshold(
    y_val, predict_prob(transformer_model, X_val_t)
)

In [ ]:
import shap
import matplotlib.pyplot as plt

background = X_train_t[:min(100, len(X_train_t))]
explain_data = X_test_t[:min(100, len(X_test_t))]

explainer = shap.GradientExplainer(
    best_mlp, background
)

shap_values = explainer.shap_values(explain_data)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.asarray(shap_values)
if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
    shap_values = shap_values[..., 0]
shap.summary_plot(
    shap_values,
    explain_data.cpu().numpy(),
    feature_names=list(ind.columns)
)
importance = np.abs(shap_values).mean(axis=0)
shap_ranking = pd.DataFrame({
    "Feature": list(ind.columns),
    "Mean absolute SHAP": importance
}).sort_values("Mean absolute SHAP", ascending=False)
print(shap_ranking)

In [ ]:
models = [
    ("Tuned MLP", best_mlp, mlp_threshold),
    ("Dual Encoder", dual_model, dual_threshold),
    ("Attention MLP", attention_model, attention_threshold),
    ("Transformer", transformer_model, transformer_threshold)
]

results = []

for name, model, threshold in models:
    results.append(
        evaluate_model(name, model, threshold)
    )

results_df = pd.DataFrame(results).sort_values(
    "ROC-AUC", ascending=False
)

print(results_df.round(4).to_string(index=False))